# Import libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import re

# Path

In [ ]:
list_judges = ['openai/gpt-oss-20b', 'llama-3.3-70b-versatile', 'moonshotai/kimi-k2-instruct', 'qwen/qwen3-32b', 'tinyllama']

In [ ]:
def extract_model_name(model_string):
  match = re.match(r'[^/-]+', model_string)
  if match:
    return match.group(0)
  else:
    return model_string

In [ ]:
# Create list of dicts with judge name and corresponding input file path
judges_name_and_path = []
for judge in list_judges:
  judge_name = extract_model_name(judge)
  new_judge = {
      "name": judge_name,
      "path": f'drive/MyDrive/ViJudge/data/position_bias/{judge_name}_position_bias_for_pair_llama_deepseek.csv'
  }
  judges_name_and_path.append(new_judge)

In [ ]:
judges_name_and_path

In [ ]:
output_path = 'drive/MyDrive/ViJudge/data/position_bias/statistics/'

# Statistics

Compare the judgments produced under the two orderings and group the outcomes into four categories. An instance is counted as:
- **Consistency** if the judge identifies the same underlying preference across both runs, including the case where both judgments are ties.
- **Biased toward first** if the judge tends to favor the response appearing in the first position across the two orderings. 
- **Biased toward second** if the judge tends to favor the response appearing in the second position.
- **Error** if at least one run does not return a valid label among A, B, or C.

In [ ]:
def cal_measures_attempt(df: pd.DataFrame):
  list_id = df['question_id'].unique()
  n_consistency = 0
  n_biased_toward_first = 0
  n_biased_toward_second = 0
  n_error = 0
  for question_id in list_id:
    question_df = df[df['question_id'] == question_id]
    if 'Unknown' in question_df['verdict'].unique():
      n_error += 1
      continue
    else:
      first_ans = question_df[question_df['position_config'] == 'original']['verdict'].iloc[0] # Extract scalar value
      second_ans = question_df[question_df['position_config'] == 'swapped']['verdict'].iloc[0] # Extract scalar value
      if first_ans != second_ans and first_ans != 'C' and second_ans != 'C':
        n_consistency += 1
      elif first_ans == 'C' and second_ans == 'C':
        n_consistency += 1
      elif first_ans == 'A' and second_ans == 'A':
        n_biased_toward_first += 1
      elif first_ans == 'B' and second_ans == 'B':
        n_biased_toward_second += 1
      elif 'C' in [first_ans, second_ans]:
        if 'A' in [first_ans, second_ans]:
          n_biased_toward_first += 1
        elif 'B' in [first_ans, second_ans]:
          n_biased_toward_second += 1

  print(f"Position bias for attempt {df['attempt'].unique()}:")
  print(f"Total consistency: {n_consistency}")
  print(f"Total biased toward first: {n_biased_toward_first}")
  print(f"Total biased toward second: {n_biased_toward_second}")
  print(f"Total error: {n_error}")
  print('*'*50)
  consistency_rate = n_consistency / len(list_id)
  biased_toward_first_rate = n_biased_toward_first / len(list_id)
  biased_toward_second_rate = n_biased_toward_second / len(list_id)
  error_rate = n_error / len(list_id)
  return consistency_rate, biased_toward_first_rate, biased_toward_second_rate, error_rate

In [ ]:
bias_result = []

for judge in judges_name_and_path:
  df = pd.read_csv(judge['path'])

  print(f"Judge: {judge['name']}")
  print(f"Total id: {len(df['question_id'].unique())}")
  print('*'*50)

  for attempt in df['attempt'].unique():
    attempt_df = df[df['attempt'] == attempt]
    # consistency_rate, biased_toward_first_rate, biased_toward_second_rate, error_rate
    cr, btfr, btsr, er = cal_measures_attempt(attempt_df)
    new_row = {
        "judge": judge['name'],
        "consistency": cr,
        "biased_toward_first": btfr,
        "biased_toward_second": btsr,
        "error": er,
        "attempt": attempt
    }
    bias_result.append(new_row)

  print('\n')
  print('='*50)
  print('\n')

# Save & show result

In [ ]:
bias_df = pd.DataFrame(bias_result)
bias_df.to_csv(output_path + 'bias_result.csv', index=False)
bias_df